# 01 — Data Cleaning & Feature Engineering: LapPick (SPK Pemilihan Laptop)

**Dataset Sumber:** [Laptop Ranked Specifications (2022) — Kaggle (`dhanushbommavaram/laptop-ranked-dataset`)](https://www.kaggle.com/datasets/dhanushbommavaram/laptop-ranked-dataset)  
**Kurs Konversi:** `1 INR = Rp 190` (Konstanta tetap sesuai dokumentasi `05_DATASET_DAN_METODE_SPK_LAPTOP.md`)  
**Tujuan Notebook:**
1. Mengunduh otomatis dataset dari Kaggle via `kagglehub` (atau unggah manual jika diperlukan).
2. Memeriksa struktur kolom asli, tipe data, duplikat, dan *missing values* (EDA awal).
3. Men-decode kolom yang ter-tokenisasi (`company`, `Type`, `Processor Brand`, `Operating System`) kembali ke label teks yang ramah pengguna.
4. Menstandarkan & membersihkan seluruh kriteria MCDM (**C1 s.d. C8** + **C9 Refresh Rate**) serta mengonversi harga INR ke Rupiah (`price_idr`).
5. Menjalankan *Sanity Check* (`assert`) dan mengekspor:
   - `laptops_clean.csv` (siap di-seed ke Laravel & dibaca oleh Streamlit)
   - `data_dictionary.md`
   - `cleaning_log.json`
   - `lappick_colab_01_outputs.zip` (otomatis terunduh di akhir notebook)

In [1]:
# === SEL 1: KONFIGURASI & IMPORT LIBRARY ===
import os
import re
import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

# Konstanta Kurs & Bobot Storage (terdokumentasi di 05_DATASET_DAN_METODE_SPK_LAPTOP.md)
INR_TO_IDR_RATE = 190
SSD_MULTIPLIER = 1.0
NON_SSD_MULTIPLIER = 0.6  # Penalti efektif jika laptop belum menggunakan SSD

OUTPUT_DIR = Path('output_01')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Konfigurasi siap. Output akan disimpan di:', OUTPUT_DIR.resolve())

Konfigurasi siap. Output akan disimpan di: /content/output_01


In [2]:
# === SEL 2: UNDUH / MUAT DATASET DARI KAGGLE ===
# Menggunakan kagglehub (tidak memerlukan API key untuk dataset publik di Google Colab).
# Jika gagal atau dijalankan offline, akan meminta upload file CSV secara manual.

csv_path = None

try:
    import kagglehub
    dl_dir = Path(kagglehub.dataset_download('dhanushbommavaram/laptop-ranked-dataset'))
    csv_candidates = list(dl_dir.rglob('*.csv'))
    if csv_candidates:
        csv_path = csv_candidates[0]
        print(f'Berhasil mengunduh otomatis via kagglehub: {csv_path}')
except Exception as e:
    print('Info kagglehub:', e)

if csv_path is None:
    local_candidates = list(Path('.').rglob('*.csv'))
    local_candidates = [p for p in local_candidates if 'laptops_clean' not in p.name]
    if local_candidates:
        csv_path = local_candidates[0]
        print(f'Menggunakan file CSV lokal: {csv_path}')
    else:
        from google.colab import files
        print('Silakan pilih file CSV dari dataset dhanushbommavaram/laptop-ranked-dataset:')
        uploaded = files.upload()
        csv_path = Path(list(uploaded.keys())[0])

df_raw = pd.read_csv(csv_path)
print(f'Dataset dimuat: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom')
df_raw.head(3)

100%|██████████| 98.9k/98.9k [00:00<00:00, 38.3MB/s]

Extracting files...
Berhasil mengunduh otomatis via kagglehub: /root/.cache/kagglehub/datasets/dhanushbommavaram/laptop-ranked-dataset/versions/3/Laptop ranked dataset.csv
Dataset dimuat: 984 baris x 27 kolom


,index,link,name,user rating,Price (in Indian Rupees),Type,Dedicated Graphic Memory Capacity,Processor Brand,SSD,RAM (in GB),RAM Type,Expandable Memory,Operating System,Touchscreen,Screen Size (in inch),Weight (in kg),Refresh Rate,screen_resolution,company,Storage,Processor name,CPU_ranking,battery_backup,gpu name,gpu_benchmark,ram_type_tokenized,gpu_processor tokenized
0,0,https://www.flipkart.com/asus-rog-strix-scar-1...,asus rog strix scar 17 core i9 12th gen - (32 ...,5.0,234990,1,8.0,1,1,32,ddr5,1,1,0,17.3,2.90,60,1440,1,1000,intel core i9 12th gen,4,4.0,nvidia geforce rtx 3070 ti,168.0,7,1
1,1,https://www.flipkart.com/asus-rog-strix-scar-1...,asus rog strix scar 15 core i9 12th gen - (32 ...,NaN,229990,1,8.0,1,1,32,ddr5,1,1,0,15.6,2.30,60,1440,1,1000,intel core i9 12th gen,4,4.0,nvidia geforce rtx 3070 ti,168.0,7,1
2,2,https://www.flipkart.com/hp-victus-ryzen-7-oct...,hp victus ryzen 7 octa core 5800h - (16 gb/512...,NaN,104091,1,4.0,2,1,16,ddr4,0,1,0,16.1,2.48,60,1080,2,512,amd ryzen 7 octa core,64,4.0,nvidia geforce rtx 3050,72.6,4,2


In [3]:
# === SEL 3: INSPEKSI KOLOM, TIPE DATA & MISSING VALUES ===
print('=== DAFTAR KOLOM ASLI ===')
for idx, col in enumerate(df_raw.columns):
    null_cnt = int(df_raw[col].isna().sum())
    sample_val = df_raw[col].dropna().iloc[0] if df_raw[col].dropna().shape[0] > 0 else None
    print(f'{idx+1:02d}. {col!r:38s} | dtype={str(df_raw[col].dtype):8s} | null={null_cnt:4d} | contoh={sample_val}')


=== DAFTAR KOLOM ASLI ===
01. 'index'                                | dtype=int64    | null=   0 | contoh=0
02. 'link'                                 | dtype=object   | null=   0 | contoh=https://www.flipkart.com/asus-rog-strix-scar-17-core-i9-12th-gen-32-gb-1-tb-ssd-windows-11-home-8-gb-graphics-nvidia-geforce-rtx-3070-ti-g733zw-ll139ws-gaming-laptop/p/itm0e721469cc7cc?pid=COMGCZUJCBRGXYFX&lid=LSTCOMGCZUJCBRGXYFXT7VNUU&marketplace=FLIPKART&store=6bo%2Fb5g&srno=b_4_73&otracker=browse&otracker1=hp_rich_navigation_PINNED_neo%2Fmerchandising_NA_NAV_EXPANDABLE_navigationCard_cc_8_L1_view-all&iid=ea708dc0-b248-4808-91aa-11961fc788e8.COMGCZUJCBRGXYFX.SEARCH&ssid=3s0bamxoztl6qzgg1653219740289
03. 'name'                                 | dtype=object   | null=   0 | contoh=asus rog strix scar 17 core i9 12th gen - (32 gb/1 tb ssd/windows 11 home/8 gb graphics/nvidia geforce rtx 3070 ti) g733zw-ll139ws gaming laptop  (17.3 inch, off black, 2.90 kg, with ms office)
04. 'user rating'           

In [4]:
# === SEL 4: PEMETAAN TOKEN KE LABEL TEKS (SESUAI DOKUMENTASI KAGGLE) ===
COMPANY_MAP = {
    1: 'ASUS', 2: 'HP', 3: 'Lenovo', 4: 'Dell', 5: 'MSI', 6: 'Realme',
    7: 'Avita', 8: 'Acer', 9: 'Samsung', 10: 'Infinix', 11: 'LG', 12: 'Apple',
    13: 'Nokia', 14: 'RedmiBook', 15: 'Mi', 16: 'VAIO'
}

TYPE_MAP = {
    1: 'Gaming Laptop',
    2: 'Thin and Light Laptop',
    3: '2 in 1 Laptop',
    4: 'Notebook',
    5: 'Laptop',
    6: '2 in 1 Gaming Laptop',
    7: 'Business Laptop',
    8: 'Chromebook',
    9: 'Creator Laptop'
}

PROC_BRAND_MAP = {
    1: 'Intel', 2: 'AMD', 3: 'Qualcomm', 4: 'Apple', 5: 'MediaTek'
}

OS_MAP = {
    1: 'Windows', 2: 'Chrome OS', 3: 'DOS', 4: 'macOS', 5: 'Ubuntu'
}

def find_col(df, candidates):
    """Mencari nama kolom yang cocok secara case-insensitive / stripped."""
    norm_map = {re.sub(r'\s+', ' ', c.strip().lower()): c for c in df.columns}
    for cand in candidates:
        key = re.sub(r'\s+', ' ', cand.strip().lower())
        if key in norm_map:
            return norm_map[key]
    # Partial match fallback
    for cand in candidates:
        key = cand.strip().lower()
        for k, orig in norm_map.items():
            if key in k:
                return orig
    raise KeyError(f'Tidak menemukan kolom dari kandidat: {candidates}')

def to_numeric_clean(series):
    """Mengekstrak angka float dari kolom numerik/string dengan aman."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    extracted = series.astype(str).str.replace(',', '', regex=False).str.extract(r'([-+]?\d*\.?\d+)')[0]
    return pd.to_numeric(extracted, errors='coerce')

def decode_token_or_str(series, mapping, fallback='Unknown'):
    def _conv(val):
        if pd.isna(val):
            return fallback
        try:
            iv = int(float(val))
            if iv in mapping:
                return mapping[iv]
        except Exception:
            pass
        s = str(val).strip()
        return s if s else fallback
    return series.apply(_conv)


In [6]:
# === SEL 5: CLEANING, DECODING & FEATURE ENGINEERING (TOLERAN NAMA KOLOM) ===
def find_col_safe(df, candidates, required=True):
    norm_map = {re.sub(r'[\s_]+', ' ', c.strip().lower()): c for c in df.columns}
    for cand in candidates:
        key = re.sub(r'[\s_]+', ' ', cand.strip().lower())
        if key in norm_map:
            return norm_map[key]
    for cand in candidates:
        key = re.sub(r'[\s_]+', ' ', cand.strip().lower())
        for k, orig in norm_map.items():
            if key in k or k in key:
                return orig
    if required:
        raise KeyError(f'Tidak menemukan kolom dari kandidat: {candidates}. Kolom yang ada: {list(df.columns)}')
    return None

RAM_TYPE_TOKEN_TO_LABEL = {
    8: 'LPDDR5 / Unified', 7: 'DDR5', 6: 'LPDDR4X', 5: 'LPDDR4',
    4: 'DDR4', 2: 'LPDDR3', 1: 'DDR3'
}

col_name = find_col_safe(df_raw, ['name', 'laptop name', 'model'])
col_link = find_col_safe(df_raw, ['link', 'product link', 'url'], required=False)
col_company = find_col_safe(df_raw, ['company', 'brand'])
col_type = find_col_safe(df_raw, ['Type', 'laptop type'])
col_os = find_col_safe(df_raw, ['Operating System', 'os'])
col_proc_brand = find_col_safe(df_raw, ['Processor Brand', 'cpu brand'])
col_proc_name = find_col_safe(df_raw, ['Processor name', 'processor', 'cpu name'])
col_gpu_name = find_col_safe(df_raw, ['gpu name', 'gpu_processor', 'gpu'])
col_price_inr = find_col_safe(df_raw, ['Price (in Indian Rupees)', 'price'])
col_ram_gb = find_col_safe(df_raw, ['RAM (in GB)', 'ram gb', 'ram'])
col_ram_type = find_col_safe(df_raw, ['RAM Type', 'ram_type'], required=False)
col_ram_token = find_col_safe(df_raw, ['ram_type tokenized', 'ram_type_tokenized', 'ram tokenized'], required=False)
col_storage = find_col_safe(df_raw, ['Storage', 'storage capacity'])
col_ssd = find_col_safe(df_raw, ['SSD'], required=False)
col_expandable = find_col_safe(df_raw, ['Expandable Memory', 'expandable'], required=False)
col_vram = find_col_safe(df_raw, ['Dedicated Graphic Memory Capacity', 'dedicated graphic', 'vram'], required=False)
col_cpu_rank = find_col_safe(df_raw, ['CPU_ranking', 'cpu ranking', 'processor ranking'])
col_gpu_bench = find_col_safe(df_raw, ['gpu_benchmark', 'gpu benchmark'])
col_battery = find_col_safe(df_raw, ['battery_backup', 'battery life', 'battery'])
col_weight = find_col_safe(df_raw, ['Weight (in kg)', 'weight'])
col_rating = find_col_safe(df_raw, ['user rating', 'rating'])
col_refresh = find_col_safe(df_raw, ['Refresh Rate', 'refresh_rate'], required=False)
col_screen_size = find_col_safe(df_raw, ['Screen Size (in inch)', 'screen size', 'inches'], required=False)
col_screen_res = find_col_safe(df_raw, ['screen_resolution', 'screen resolution', 'resolution'], required=False)
col_touch = find_col_safe(df_raw, ['Touchscreen', 'touch screen'], required=False)

df = pd.DataFrame()
df['name'] = df_raw[col_name].astype(str).str.strip()
df['brand'] = decode_token_or_str(df_raw[col_company], COMPANY_MAP)
df['type_name'] = decode_token_or_str(df_raw[col_type], TYPE_MAP)
df['os'] = decode_token_or_str(df_raw[col_os], OS_MAP)
df['processor_brand'] = decode_token_or_str(df_raw[col_proc_brand], PROC_BRAND_MAP)
df['cpu_name'] = df_raw[col_proc_name].astype(str).str.strip()
df['gpu_name'] = df_raw[col_gpu_name].astype(str).str.strip()

if col_ram_type:
    df['ram_type'] = decode_token_or_str(df_raw[col_ram_type], RAM_TYPE_TOKEN_TO_LABEL, fallback='DDR4')
else:
    df['ram_type'] = 'DDR4'

if col_ram_token:
    df['ram_type_score'] = to_numeric_clean(df_raw[col_ram_token]).fillna(4.0)
elif col_ram_type and pd.api.types.is_numeric_dtype(df_raw[col_ram_type]):
    df['ram_type_score'] = to_numeric_clean(df_raw[col_ram_type]).fillna(4.0)
else:
    df['ram_type_score'] = 4.0

df['has_ssd'] = (to_numeric_clean(df_raw[col_ssd]).fillna(1) > 0).astype(int) if col_ssd else 1
df['expandable_memory'] = (to_numeric_clean(df_raw[col_expandable]).fillna(0) > 0).astype(int) if col_expandable else 0
df['touchscreen'] = (to_numeric_clean(df_raw[col_touch]).fillna(0) > 0).astype(int) if col_touch else 0
df['dedicated_vram_gb'] = to_numeric_clean(df_raw[col_vram]).fillna(0.0) if col_vram else 0.0
df['screen_inches'] = to_numeric_clean(df_raw[col_screen_size]).fillna(15.6) if col_screen_size else 15.6
df['screen_resolution'] = df_raw[col_screen_res].astype(str).str.strip() if col_screen_res else '1920x1080'
df['source_link'] = df_raw[col_link].astype(str).str.strip() if col_link else ''

# === KOLOM KRITERIA MCDM (C1 s.d. C9) ===
df['price_inr'] = to_numeric_clean(df_raw[col_price_inr])
df['price_idr'] = (df['price_inr'] * INR_TO_IDR_RATE).round(0)  # C1 (Cost)
df['ram_gb'] = to_numeric_clean(df_raw[col_ram_gb])              # C2 (Benefit)

raw_storage = to_numeric_clean(df_raw[col_storage])
df['storage_gb'] = np.where(raw_storage <= 8, raw_storage * 1024, raw_storage)
df['storage_effective_gb'] = np.where(
    df['has_ssd'] == 1,
    df['storage_gb'] * SSD_MULTIPLIER,
    df['storage_gb'] * NON_SSD_MULTIPLIER
)  # C3 (Benefit)

df['cpu_ranking'] = to_numeric_clean(df_raw[col_cpu_rank])       # C4 (Cost: semakin kecil semakin bagus)
df['gpu_benchmark'] = to_numeric_clean(df_raw[col_gpu_bench])    # C5 (Benefit: semakin besar semakin bagus)
df['battery_hours'] = to_numeric_clean(df_raw[col_battery])      # C6 (Benefit)
df['weight_kg'] = to_numeric_clean(df_raw[col_weight])           # C7 (Cost)
df['user_rating'] = to_numeric_clean(df_raw[col_rating])         # C8 (Benefit)
df['refresh_rate_hz'] = to_numeric_clean(df_raw[col_refresh]) if col_refresh else 60.0  # C9 (Benefit)

print('Ringkasan missing value sebelum filtering:')
criteria_cols = [
    'price_idr', 'ram_gb', 'storage_effective_gb', 'cpu_ranking',
    'gpu_benchmark', 'battery_hours', 'weight_kg', 'user_rating', 'refresh_rate_hz'
]
print(df[criteria_cols].isna().sum())

Ringkasan missing value sebelum filtering:
price_idr                 0
ram_gb                    0
storage_effective_gb      0
cpu_ranking               0
gpu_benchmark             0
battery_hours             0
weight_kg                 0
user_rating             294
refresh_rate_hz           0
dtype: int64


In [7]:
# === SEL 6: PENANGANAN DUPLIKAT & MISSING VALUES SECARA TRANSPARAN ===
# Aturan sesuai 04_AGENTS_SPK_LAPTOP.md:
# 1. Tidak ada silent fallback. Semua langkah dicatat di cleaning_log.json.
# 2. Untuk refresh_rate_hz yang kosong/0, standar industri layar laptop non-gaming adalah 60 Hz (dicatat jumlahnya).
# 3. Baris yang masih memiliki missing value atau nilai <= 0 pada kriteria inti dihapus secara eksplisit.

initial_rows = len(df)

# Hapus duplikat berdasarkan link atau kombinasi (name, price_inr, ram_gb, storage_gb)
df_dedup = df.drop_duplicates(subset=['name', 'price_inr', 'ram_gb', 'storage_gb', 'cpu_name']).copy()
duplicates_removed = initial_rows - len(df_dedup)

# Standarisasi refresh_rate_hz jika kosong atau <= 0 menjadi 60 Hz (dicatat eksplisit)
refresh_missing_mask = df_dedup['refresh_rate_hz'].isna() | (df_dedup['refresh_rate_hz'] <= 0)
refresh_defaulted_count = int(refresh_missing_mask.sum())
df_dedup.loc[refresh_missing_mask, 'refresh_rate_hz'] = 60.0

# Cek missing value pada 8 kriteria lainnya
core_criteria = [
    'price_idr', 'ram_gb', 'storage_effective_gb', 'cpu_ranking',
    'gpu_benchmark', 'battery_hours', 'weight_kg', 'user_rating'
]

missing_by_col = {col: int(df_dedup[col].isna().sum()) for col in core_criteria}

# Jika user_rating atau battery_hours memiliki banyak NaN, kita catat dan tangani secara terdokumentasi:
# - Hanya baris dengan spesifikasi inti valid yang dipertahankan.
# - Jika user_rating atau battery_hours NaN < 30% data, drop baris NaN agar 100% data riil tanpa imputasi.
# - Jika NaN >= 30%, gunakan median per type_name agar katalog tidak habis, dan beri flag `is_imputed`.
df_dedup['is_imputed'] = 0
imputation_details = {}

for col in ['user_rating', 'battery_hours', 'weight_kg']:
    na_cnt = int(df_dedup[col].isna().sum())
    na_ratio = na_cnt / max(len(df_dedup), 1)
    if na_cnt > 0 and na_ratio >= 0.25:
        df_dedup['is_imputed'] = np.where(df_dedup[col].isna(), 1, df_dedup['is_imputed'])
        medians = df_dedup.groupby('type_name')[col].transform('median')
        global_med = float(df_dedup[col].median())
        df_dedup[col] = df_dedup[col].fillna(medians).fillna(global_med)
        imputation_details[col] = {'method': 'group_median_by_type_name', 'count': na_cnt, 'global_median': global_med}
    else:
        imputation_details[col] = {'method': 'dropped_rows_with_na', 'count': na_cnt}

# Drop baris yang masih NaN atau <= 0 pada kriteria wajib
valid_mask = pd.Series(True, index=df_dedup.index)
for col in criteria_cols:
    valid_mask &= df_dedup[col].notna() & (df_dedup[col] > 0)

df_clean = df_dedup[valid_mask].copy().reset_index(drop=True)
df_clean.insert(0, 'id', np.arange(1, len(df_clean) + 1))
df_clean['price_idr'] = df_clean['price_idr'].astype(np.int64)
df_clean['ram_gb'] = df_clean['ram_gb'].astype(int)
df_clean['storage_gb'] = df_clean['storage_gb'].astype(int)
df_clean['cpu_ranking'] = df_clean['cpu_ranking'].astype(int)
df_clean['refresh_rate_hz'] = df_clean['refresh_rate_hz'].astype(int)

rows_dropped_invalid = len(df_dedup) - len(df_clean)

# === SANITY CHECKS (ASSERT) ===
assert len(df_clean) >= 300, f'Jumlah baris bersih terlalu sedikit: {len(df_clean)}'
assert df_clean[criteria_cols].isna().sum().sum() == 0, 'Masih ada nilai NaN pada kolom kriteria!'
assert (df_clean[criteria_cols] <= 0).sum().sum() == 0, 'Masih ada nilai <= 0 pada kolom kriteria!'

print(f'Baris awal            : {initial_rows}')
print(f'Duplikat dihapus      : {duplicates_removed}')
print(f'Baris invalid dihapus : {rows_dropped_invalid}')
print(f'Baris bersih akhir    : {len(df_clean)}')
df_clean.head(5)

Baris awal            : 984
Duplikat dihapus      : 35
Baris invalid dihapus : 0
Baris bersih akhir    : 949


,id,name,brand,type_name,os,processor_brand,cpu_name,gpu_name,ram_type,ram_type_score,has_ssd,expandable_memory,touchscreen,dedicated_vram_gb,screen_inches,screen_resolution,source_link,price_inr,price_idr,ram_gb,storage_gb,storage_effective_gb,cpu_ranking,gpu_benchmark,battery_hours,weight_kg,user_rating,refresh_rate_hz,is_imputed
0,1,asus rog strix scar 17 core i9 12th gen - (32 ...,ASUS,Gaming Laptop,Windows,Intel,intel core i9 12th gen,nvidia geforce rtx 3070 ti,ddr5,7,1,1,0,8.0,17.3,1440,https://www.flipkart.com/asus-rog-strix-scar-1...,234990,44648100,32,1000,1000.0,4,168.00,4.0,2.900000,5.0,60,0
1,2,asus rog strix scar 15 core i9 12th gen - (32 ...,ASUS,Gaming Laptop,Windows,Intel,intel core i9 12th gen,nvidia geforce rtx 3070 ti,ddr5,7,1,1,0,8.0,15.6,1440,https://www.flipkart.com/asus-rog-strix-scar-1...,229990,43698100,32,1000,1000.0,4,168.00,4.0,2.300000,4.5,60,1
2,3,hp victus ryzen 7 octa core 5800h - (16 gb/512...,HP,Gaming Laptop,Windows,AMD,amd ryzen 7 octa core,nvidia geforce rtx 3050,ddr4,4,1,0,0,4.0,16.1,1080,https://www.flipkart.com/hp-victus-ryzen-7-oct...,104091,19777290,16,512,512.0,64,72.60,4.0,2.480000,4.5,60,1
3,4,lenovo ideapad gaming 3i ryzen 7 octa core r7-...,Lenovo,Gaming Laptop,Windows,AMD,amd ryzen 7 octa core 5th gen,nvidia geforce rtx 3050,ddr4,4,1,0,0,4.0,15.6,1080,https://www.flipkart.com/lenovo-ideapad-gaming...,87717,16666230,16,512,512.0,24,72.60,4.0,2.250000,4.5,60,1
4,5,lenovo yoga slim 7 core i5 11th gen - (16 gb/5...,Lenovo,Thin and Light Laptop,Windows,Intel,intel core i5 11th gen,intel hd,ddr4,4,1,0,1,0.0,14.0,1080,https://www.flipkart.com/lenovo-yoga-slim-7-co...,75990,14438100,16,512,512.0,108,4.23,4.5,1.500672,4.3,60,1


In [8]:
# === SEL 7: EKSPOR FILE OUTPUT & RINGKASAN UNTUK DIKIRIM BALIK ===
clean_csv_path = OUTPUT_DIR / 'laptops_clean.csv'
df_clean.to_csv(clean_csv_path, index=False)

cleaning_log = {
    'dataset_ref': 'dhanushbommavaram/laptop-ranked-dataset',
    'inr_to_idr_rate': INR_TO_IDR_RATE,
    'ssd_multiplier': SSD_MULTIPLIER,
    'non_ssd_multiplier': NON_SSD_MULTIPLIER,
    'initial_rows': int(initial_rows),
    'duplicates_removed': int(duplicates_removed),
    'rows_dropped_invalid_or_na': int(rows_dropped_invalid),
    'final_clean_rows': int(len(df_clean)),
    'refresh_rate_defaulted_to_60hz_count': int(refresh_defaulted_count),
    'missing_before_handling': missing_by_col,
    'imputation_details': imputation_details,
    'brands_distribution': df_clean['brand'].value_counts().to_dict(),
    'types_distribution': df_clean['type_name'].value_counts().to_dict(),
    'criteria_summary': {
        col: {
            'min': float(df_clean[col].min()),
            'median': float(df_clean[col].median()),
            'mean': round(float(df_clean[col].mean()), 4),
            'max': float(df_clean[col].max())
        } for col in criteria_cols
    }
}

log_path = OUTPUT_DIR / 'cleaning_log.json'
log_path.write_text(json.dumps(cleaning_log, indent=2), encoding='utf-8')

# Buat data_dictionary.md otomatis
dict_lines = [
    '# Data Dictionary — `laptops_clean.csv`',
    '',
    f'- **Jumlah Baris Bersih:** {len(df_clean)}',
    f'- **Kurs Konversi:** 1 INR = Rp {INR_TO_IDR_RATE}',
    '',
    '| Kolom | Tipe | Peran SPK | Min | Median | Max |',
    '|---|---|---|---|---|---|'
]
role_map = {
    'price_idr': 'C1 Harga (Cost)',
    'ram_gb': 'C2 RAM (Benefit)',
    'storage_effective_gb': 'C3 Storage Efektif (Benefit)',
    'cpu_ranking': 'C4 Peringkat CPU (Cost - kecil lebih baik)',
    'gpu_benchmark': 'C5 Skor GPU (Benefit)',
    'battery_hours': 'C6 Daya Tahan Baterai (Benefit)',
    'weight_kg': 'C7 Bobot (Cost)',
    'user_rating': 'C8 Rating Pengguna (Benefit)',
    'refresh_rate_hz': 'C9 Refresh Rate Layar (Benefit)'
}
for col in df_clean.columns:
    role = role_map.get(col, 'Atribut Katalog / Filter')
    if col in criteria_cols:
        s = cleaning_log['criteria_summary'][col]
        dict_lines.append(f'| `{col}` | `{df_clean[col].dtype}` | {role} | {s["min"]} | {s["median"]} | {s["max"]} |')
    else:
        dict_lines.append(f'| `{col}` | `{df_clean[col].dtype}` | {role} | - | - | - |')

dict_path = OUTPUT_DIR / 'data_dictionary.md'
dict_path.write_text('\n'.join(dict_lines), encoding='utf-8')

# Buat paket ZIP agar mudah diunduh 1 kali
zip_path = Path('lappick_colab_01_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(clean_csv_path, arcname='laptops_clean.csv')
    zf.write(log_path, arcname='cleaning_log.json')
    zf.write(dict_path, arcname='data_dictionary.md')

print('=== COPY RINGKASAN DI BAWAH INI KE CHAT ANTIGRAVITY ===')
print(json.dumps(cleaning_log, indent=2))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print(f'File ZIP tersimpan di: {zip_path.resolve()}')

=== COPY RINGKASAN DI BAWAH INI KE CHAT ANTIGRAVITY ===
{
  "dataset_ref": "dhanushbommavaram/laptop-ranked-dataset",
  "inr_to_idr_rate": 190,
  "ssd_multiplier": 1.0,
  "non_ssd_multiplier": 0.6,
  "initial_rows": 984,
  "duplicates_removed": 35,
  "rows_dropped_invalid_or_na": 0,
  "final_clean_rows": 949,
  "refresh_rate_defaulted_to_60hz_count": 0,
  "missing_before_handling": {
    "price_idr": 0,
    "ram_gb": 0,
    "storage_effective_gb": 0,
    "cpu_ranking": 0,
    "gpu_benchmark": 0,
    "battery_hours": 0,
    "weight_kg": 0,
    "user_rating": 277
  },
  "imputation_details": {
    "user_rating": {
      "method": "group_median_by_type_name",
      "count": 277,
      "global_median": 4.3
    },
    "battery_hours": {
      "method": "dropped_rows_with_na",
      "count": 0
    },
    "weight_kg": {
      "method": "dropped_rows_with_na",
      "count": 0
    }
  },
  "brands_distribution": {
    "ASUS": 284,
    "Dell": 183,
    "Lenovo": 178,
    "HP": 147,
    "Acer": 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>